# Data Generation: Teacher Inference

Loads the fine-tuned teacher (`Qwen2.5-3B-Instruct` + LoRA adapter) and generates responses for 232 German customer-support prompts.<br>
The resulting `teacher_generated_data.json` is the training set for the student in Notebook 03.

**Pipeline position:** 01 Fine-Tuning → `[02 Data Generation]` → 03 Student Distillation → 04 Evaluation

**Strong GPU required.** This notebook was developed on a Kaggle T4 (16 GB VRAM).<br>
[![Open Notebook in Kaggle](https://img.shields.io/badge/Open%20Notebook%20in-Kaggle-20BEFF?style=for-the-badge&logo=kaggle&logoColor=white)](https://www.kaggle.com/code/dennisfeyerabend/02-data-generation)

## 1. Setup

Install dependencies and check GPU.<br>
Local users: skip the pip cell — install via `pip install -r requirements.txt` instead.

In [1]:
%%capture
!pip install -q --upgrade unsloth trl datasets transformers peft bitsandbytes accelerate huggingface_hub python-dotenv

In [2]:
!nvidia-smi
import torch
print(f"PyTorch Version:    {torch.__version__}")
print(f"CUDA Available:     {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:            {torch.cuda.get_device_name(0)}")
    print(f"VRAM:           {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Fri May 15 16:48:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   69C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Authenticate

**Kaggle:** add `HF_TOKEN` via *Add-ons → Secrets*.<br>
**Local:** create a `.env` file with `HF_TOKEN=your_token`.

In [3]:
import os
from dotenv import load_dotenv

load_dotenv()  # loads .env locally, no-op on Kaggle

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle Secrets.")
except ImportError:
    print("kaggle_secrets not available — using .env fallback.")
except Exception as e:
    print(f"Kaggle Secrets found but could not load HF_TOKEN: {e}")

HF_TOKEN = os.getenv("HF_TOKEN")

if HF_TOKEN:
    print(f"HF_TOKEN set. ({HF_TOKEN[:4]}...{HF_TOKEN[-4:]})")
else:
    print("WARNING: HF_TOKEN is not set. The save cell will fail.")

HF_TOKEN loaded from Kaggle Secrets.
HF_TOKEN set. (hf_R...nFKm)


## 3. Load Teacher Model

The teacher is `Qwen2.5-3B-Instruct-bnb-4bit` with the LoRA adapter from Notebook 01 applied on top.   
The adapter is loaded directly from Hugging Face Hub — no local files required.

In [4]:
from unsloth import FastLanguageModel
from peft import PeftModel
import torch

MAX_SEQ_LENGTH = 2048
ADAPTER_REPO = "Feyerade/german-support-qwen-lora-adapter"

# Load 3B base model in 4-bit quantisation (~2 GB VRAM)
base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

# Apply fine-tuned adapter — unload first if cell is re-run
if "model" in dir() and isinstance(model, PeftModel):
    model = model.unload()

model = PeftModel.from_pretrained(base_model, ADAPTER_REPO)
print(f"Adapter applied: {isinstance(model, PeftModel)}")

# Enable Unsloth inference kernels (~2x faster generation, no gradient tracking)
FastLanguageModel.for_inference(model)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

unsloth/Qwen2.5-3B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Adapter applied: True


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 2048, padding_idx=151665)
        (layers): ModuleList(
          (0-35): 36 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

## 4. Define Prompts

232 German customer-support queries covering shipping, returns, account issues, payments, and more.   
These are the inputs the teacher answers — the resulting (prompt, response) pairs become the student's training data.

In [5]:
SYSTEM_PROMPT = (
    "Du bist ein professioneller Kundenservice-Mitarbeiter. "
    "Antworte freundlich, loesungsorientiert und auf Deutsch. "
    "Halte deine Antworten unter 150 Woertern."
)

prompts = [
    "Meine Lieferung sollte gestern ankommen, aber im Tracking steht immer noch 'in Bearbeitung'.",
    "Ich kann mich nicht mehr in mein Konto einloggen. Der Code per Email kommt nicht an.",
    "Warum wurde meine Bestellung ohne Grund storniert?",
    "Ich habe eine Versandbestaetigung bekommen, aber keine Sendungsnummer.",
    "Kann ich meine Bestellung noch an eine andere Adresse schicken lassen?",
    "Ich habe aus Versehen zwei Mal bestellt. Koennen Sie eine Bestellung stornieren?",
    "Der Preis im Warenkorb ist hoeher als auf der Produktseite.",
    "Warum wurde meine Zahlung abgelehnt obwohl genug Geld auf der Karte ist?",
    "Ich finde meine alte Bestellung in meinem Konto nicht mehr.",
    "Wie lange dauert der Versand normalerweise?",
    "Ich habe keine Bestaetigungsmail fuer meine Bestellung erhalten.",
    "Der Rabattcode aus dem Newsletter funktioniert nicht.",
    "Kann ich meine Bestellung auch an eine Packstation liefern lassen?",
    "Meine Bestellung ist laut Tracking zugestellt, aber ich habe nichts erhalten.",
    "Wie kann ich meine Zahlungsmethode aendern?",
    "Ich moechte eine Kopie meiner letzten Rechnung herunterladen.",
    "Warum wurde meine Bestellung in zwei Paketen verschickt?",
    "Das Produkt sieht anders aus als auf der Website.",
    "Ich kann meine Bestellung nicht abschliessen. Der Bezahlbutton reagiert nicht.",
    "Warum sind die Versandkosten ploetzlich hoeher als vorher?",
    "Ich habe eine Mahnung bekommen, obwohl ich schon bezahlt habe.",
    "Wie kann ich mein Passwort wiederherstellen wenn ich keinen Zugriff auf meine Email habe?",
    "Mein Warenkorb ist nach dem Einloggen leer.",
    "Kann ich mehrere Gutscheine gleichzeitig einloesen?",
    "Ich habe versehentlich die falsche Farbe bestellt. Kann ich das noch aendern?",
    "Der Kundenservice antwortet seit Tagen nicht auf meine Email.",
    "Ich moechte wissen wann meine Rueckerstattung bearbeitet wird.",
    "Meine Bestellung wurde als geliefert markiert, aber ich habe kein Paket bekommen.",
    "Warum funktioniert die Zahlung mit PayPal nicht?",
    "Ich moechte mein Lieferdatum verschieben.",
    "Kann ich meine Bestellung auch selbst im Laden abholen?",
    "Das Paket ist beschaedigt angekommen und der Inhalt fehlt teilweise.",
    "Ich habe ein Abo abgeschlossen, aber finde die Einstellungen nicht.",
    "Warum wird meine Kreditkarte nicht akzeptiert?",
    "Ich habe eine falsche Rechnung bekommen.",
    "Der Artikel im Paket entspricht nicht meiner Bestellung.",
    "Warum kann ich mein Konto nicht verifizieren?",
    "Ich moechte mein Passwort aendern, finde aber die Option nicht.",
    "Meine Bestellung ist seit Tagen im Status 'wird vorbereitet'.",
    "Ich habe einen Artikel zurueckgeschickt, aber noch keine Rueckerstattung erhalten.",
    "Warum kann ich keine Bewertung fuer ein Produkt abgeben?",
    "Ich habe eine Email ueber eine Bestellung erhalten die ich nicht gemacht habe.",
    "Der Support Chat laedt nicht in meinem Browser.",
    "Ich kann mein Profilbild nicht hochladen.",
    "Warum wurde mein Konto ohne Vorwarnung gesperrt?",
    "Ich finde die Option zum Abbestellen des Newsletters nicht.",
    "Die Website zeigt eine Fehlermeldung beim Checkout.",
    "Mein Gutschein wurde als benutzt markiert obwohl ich ihn nicht verwendet habe.",
    "Ich moechte meine Lieferadresse dauerhaft aendern.",
    "Warum sind manche Produkte in meinem Land nicht lieferbar?",
    "Ich habe den falschen Artikel erhalten.",
    "Meine Bestellung wurde teilweise geliefert. Wann kommt der Rest?",
    "Warum wurde meine Ruecksendung abgelehnt?",
    "Ich kann die App nicht auf meinem Handy installieren.",
    "Die App zeigt nur einen weissen Bildschirm beim Start.",
    "Ich bekomme staendig Fehlermeldungen beim Einloggen.",
    "Wie kann ich mein Kundenkonto komplett loeschen?",
    "Ich moechte meine Emailadresse im Konto aendern.",
    "Warum funktioniert die Zwei Faktor Anmeldung nicht?",
    "Ich habe meine Bestellung an die falsche Adresse geschickt.",
    "Wie kann ich einen Artikel umtauschen?",
    "Mein Paket ist im Versand verloren gegangen.",
    "Ich sehe eine unbekannte Belastung auf meiner Rechnung.",
    "Kann ich eine Bestellung pausieren bevor sie verschickt wird?",
    "Warum ist mein Konto ploetzlich deaktiviert?",
    "Ich kann keine neuen Produkte in den Warenkorb legen.",
    "Der Warenkorb aktualisiert die Menge nicht richtig.",
    "Ich habe einen Gutschein geschenkt bekommen, aber er wird nicht akzeptiert.",
    "Kann ich eine Ruecksendung ohne Originalverpackung machen?",
    "Mein Paket wurde an den Absender zurueckgeschickt.",
    "Ich finde die Rechnung fuer meine letzte Bestellung nicht.",
    "Warum ist meine Bestellung teurer als erwartet?",
    "Wie lange dauert eine Rueckerstattung normalerweise?",
    "Ich moechte wissen ob ein Produkt wieder auf Lager kommt.",
    "Warum funktioniert der Login ueber Google nicht?",
    "Ich bekomme keine SMS fuer die Anmeldung.",
    "Mein Konto zeigt eine falsche Bestellhistorie.",
    "Ich kann meine Bestellung nicht verfolgen.",
    "Warum wird mein Gutschein nicht auf reduzierte Artikel angewendet?",
    "Ich habe eine falsche Emailadresse im Konto gespeichert.",
    "Wie kann ich meine Bestellung komplett stornieren?",
    "Warum funktioniert der Warenkorb auf dem Handy nicht?",
    "Ich sehe doppelte Bestellungen in meinem Konto.",
    "Der Support Bot versteht meine Anfrage nicht.",
    "Meine Bestellung wurde zweimal berechnet.",
    "Warum kann ich kein neues Passwort setzen?",
    "Ich habe eine Lieferung an eine alte Adresse bekommen.",
    "Ich moechte mein Abo vorzeitig beenden.",
    "Warum wurde mein Gutschein deaktiviert?",
    "Ich kann meine Ruecksendung im Portal nicht anmelden.",
    "Der Artikel fehlt in meiner Lieferung.",
    "Meine Bestellung wurde automatisch storniert.",
    "Ich moechte eine Rechnung mit ausgewiesener Mehrwertsteuer.",
    "Warum funktioniert die Zahlungsseite nicht?",
    "Ich habe eine falsche Telefonnummer im Konto hinterlegt.",
    "Kann ich eine Rueckerstattung auf eine andere Zahlungsmethode bekommen?",
    "Mein Paket wurde beim Nachbarn abgegeben ohne Info.",
    "Warum bekomme ich keine Versandbestaetigung?",
    "Ich habe einen falschen Namen auf der Rechnung.",
    "Kann ich meine Bestellung beschleunigen?",
    "Der Artikel ist defekt angekommen.",
    "Warum ist mein Konto ploetzlich leer?",
    "Ich kann meine Lieferadresse nicht speichern.",
    "Die Website ist sehr langsam beim Bestellen.",
    "Mein Rabattcode ist angeblich abgelaufen.",
    "Ich moechte mein Abo auf monatliche Zahlung umstellen.",
    "Warum wird meine Bestellung immer wieder abgelehnt?",
    "Ich kann meine Zahlungsart nicht entfernen.",
    "Mein Paket steckt seit Tagen im Versandzentrum fest.",
    "Ich habe ein Produkt bestellt das jetzt ausverkauft ist.",
    "Warum bekomme ich staendig Fehlermeldungen im Konto?",
    "Ich kann meine Daten im Profil nicht bearbeiten.",
    "Mein Warenkorb verschwindet nach dem Aktualisieren.",
    "Ich moechte eine Sendung umleiten.",
    "Warum sehe ich unterschiedliche Preise fuer das gleiche Produkt?",
    "Ich habe meine Bestellung aus Versehen bestaetigt.",
    "Kann ich meine Bestellung nach dem Versand noch stornieren?",
    "Mein Gutschein funktioniert nur teilweise.",
    "Ich bekomme staendig Werbung obwohl ich mich abgemeldet habe.",
    "Wie kann ich mein Konto voruebergehend deaktivieren?",
    "Meine Rueckerstattung ist noch nicht auf dem Konto.",
    "Ich kann meine Bestellung nicht herunterladen.",
    "Warum funktioniert die Filterfunktion im Shop nicht?",
    "Ich habe eine falsche Groesse geliefert bekommen.",
    "Mein Paket wurde geoeffnet geliefert.",
    "Ich moechte meine Zahlungsart auf Rechnung aendern.",
    "Warum kann ich keine Artikel bewerten?",
    "Ich sehe eine unbekannte Bestellung in meinem Konto.",
    "Meine Bestellung ist verschwunden.",
    "Ich kann meine Telefonnummer nicht bestaetigen.",
    "Der Preis im Checkout ist falsch berechnet.",
    "Ich moechte eine Bestellung zusammenlegen.",
    "Warum funktioniert der Warenkorb nicht im Browser?",
    "Ich kann meine Ruecksendung nicht ausdrucken.",
    "Meine Lieferung kam viel spaeter als angekuendigt.",
    "Warum kann ich keinen neuen Gutschein einloesen?",
    "Ich habe keine Rechnung per Email erhalten.",
    "Der Trackinglink funktioniert nicht.",
    "Ich kann meine Bestellung nicht bearbeiten.",
    "Warum wurde mein Konto aus Sicherheitsgruenden gesperrt?",
    "Ich moechte meine Daten exportieren.",
    "Der Artikel kam ohne Zubehoer.",
    "Warum ist mein Konto ploetzlich ausgeloggt?",
    "Ich kann die App nicht aktualisieren.",
    "Meine Bestellung wurde aufgeteilt ohne Info.",
    "Ich moechte eine Teilerstattung fuer einen Artikel.",
    "Warum wurde meine Ruecksendung noch nicht bestaetigt?",
    "Ich sehe eine falsche Adresse im Konto.",
    "Mein Paket wurde beim Transport beschaedigt.",
    "Ich moechte den Versand auf Express aendern.",
    "Warum kann ich kein neues Konto erstellen?",
    "Ich bekomme keine Aktivierungsmail.",
    "Der Login Button funktioniert nicht.",
    "Ich habe eine Lieferung fuer jemand anderen erhalten.",
    "Warum wird mein Gutschein nur teilweise angewendet?",
    "Ich moechte eine Bestellung fuer meine Firma machen.",
    "Meine Bestellung wurde ohne Grund zurueckgesendet.",
    "Ich kann mein Passwort nicht zuruecksetzen.",
    "Warum funktioniert die Suche im Shop nicht?",
    "Ich sehe einen anderen Preis in der App.",
    "Meine Bestellung wurde doppelt verschickt.",
    "Ich moechte meine Bestellung splitten.",
    "Warum ist mein Konto eingeschraenkt?",
    "Ich bekomme keine Updates zum Versand.",
    "Meine Ruecksendung wurde noch nicht bearbeitet.",
    "Ich kann mein Profil nicht speichern.",
    "Warum funktioniert die Zahlungsbestaetigung nicht?",
    "Mein Paket ist leer angekommen.",
    "Ich moechte meine Bestellung an einen anderen Empfaenger schicken.",
    "Warum funktioniert der Gutscheincode nicht im Warenkorb?",
    "Ich kann meine Adresse nicht bestaetigen.",
    "Meine Bestellung wurde ohne Info geaendert.",
    "Warum sehe ich falsche Preise im Warenkorb?",
    "Ich moechte eine Bestellung verschenken.",
    "Meine Rechnung zeigt falsche Positionen.",
    "Warum kann ich mein Konto nicht loeschen?",
    "Ich habe eine Bestellung ohne Konto gemacht und finde sie nicht.",
    "Der Artikel ist ploetzlich nicht mehr verfuegbar.",
    "Ich kann meine Ruecksendung nicht verfolgen.",
    "Warum funktioniert der Login in der App nicht?",
    "Ich moechte eine Bestellung fuer spaeter planen.",
    "Mein Paket wurde im Regen abgestellt.",
    "Warum funktioniert mein Gutscheincode nur einmal?",
    "Ich sehe eine fremde Adresse im Konto.",
    "Meine Bestellung wurde ohne Zustimmung storniert.",
    "Ich kann mein Konto nicht wieder aktivieren.",
    "Warum funktioniert der Support Chat nicht?",
    "Meine Lieferung wurde an die falsche Adresse geschickt.",
    "Ich moechte meine Bestellung neu berechnen lassen.",
    "Warum kann ich keinen Rabatt anwenden?",
    "Ich sehe keine Trackinginformationen.",
    "Meine Ruecksendung wurde verloren.",
    "Ich kann meine Bestellung nicht herunterladen.",
    "Warum wird mein Warenkorb nicht gespeichert?",
    "Ich habe eine falsche Bestellbestaetigung erhalten.",
    "Mein Paket wurde im Treppenhaus abgestellt.",
    "Warum funktioniert mein Login Code nicht?",
    "Ich moechte eine Bestellung fuer jemand anderen bezahlen.",
    "Meine Rechnung zeigt eine doppelte Belastung.",
    "Warum kann ich kein neues Passwort erstellen?",
    "Ich habe eine Bestellung bekommen die ich nicht bestellt habe.",
    "Mein Gutschein wurde abgelehnt.",
    "Warum funktioniert der Checkout nicht?",
    "Ich kann meine Bestellung nicht sehen.",
    "Meine Ruecksendung wurde nicht akzeptiert.",
    "Warum kann ich meine Adresse nicht aktualisieren?",
    "Ich habe einen Artikel doppelt bekommen.",
    "Meine Bestellung wurde ohne Grund verzzoegert.",
    "Warum bekomme ich keine Email bestaetigung?",
    "Ich kann mein Konto nicht verifizieren.",
    "Meine Bestellung wurde falsch gepackt.",
    "Warum funktioniert mein Rabatt nicht?",
    "Ich sehe eine unbekannte Zahlung.",
    "Meine Lieferung fehlt komplett.",
    "Warum wurde mein Konto deaktiviert?",
    "Ich kann meine Bestellung nicht abschliessen.",
    "Meine Rueckerstattung ist zu niedrig.",
    "Warum funktioniert mein Gutschein nicht mehr?",
    "Ich sehe eine falsche Lieferadresse.",
    "Meine Bestellung ist verschwunden.",
    "Warum kann ich keine Zahlung abschliessen?",
    "Ich habe eine falsche Rechnung bekommen.",
    "Meine Lieferung ist unvollstaendig.",
    "Warum funktioniert der Login nicht mehr?",
    "Ich kann mein Passwort nicht aendern.",
    "Meine Bestellung wurde storniert ohne Info.",
    "Warum sehe ich falsche Preise?",
    "Ich kann meine Bestellung nicht verfolgen.",
    "Meine Ruecksendung wurde nicht bestaetigt.",
    "Warum funktioniert der Gutschein nicht?",
    "Ich kann mein Konto nicht finden.",
    "Meine Bestellung wurde doppelt berechnet."
]

print(f"Prompts: {len(prompts)}")

Prompts: 232


## 5. Generate Training Data

The teacher generates one response per prompt using sampling (temperature 0.7, top_p 0.9).   
Each (prompt, response) pair becomes one training example for the student.  
**Expected runtime on a Kaggle T4: ~25 minutes.**

In [6]:
import warnings
import transformers

warnings.filterwarnings("ignore", category=FutureWarning)
transformers.logging.set_verbosity_error()

generated_data = []

for i, prompt in enumerate(prompts):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id,
    )
    # Slice off the prompt tokens — decode only the generated response
    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )
    generated_data.append({"instruction": prompt, "output": response})

    if (i + 1) % 25 == 0:
        print(f"Generated: {i + 1}/{len(prompts)}")

print(f"\nDone. {len(generated_data)} responses generated.")

Generated: 25/232
Generated: 50/232
Generated: 75/232
Generated: 100/232
Generated: 125/232
Generated: 150/232
Generated: 175/232
Generated: 200/232
Generated: 225/232

Done. 232 responses generated.


## 6. Qualitative Check

The teacher model occasionally generates off-topic, incomplete, or hallucinated responses.   
A random sample gives a quick visual gut-check before the data is committed to the Hub.   
The automated check in the next step catches length-based failures — this step catches everything else.

In [7]:
import random

for item in random.sample(generated_data, 5):
    print(f"\n{'='*60}")
    print(f"Query:    {item['instruction']}")
    print(f"Response: {item['output']}")


Query:    Kann ich eine Ruecksendung ohne Originalverpackung machen?
Response: Natürlich! Wir akzeptieren Ruecksendungen auch ohne Originalverpackung. Bitte verwenden Sie saubere, unbeschnittene Papierservetten oder PVC-Folie. Das schütet den Inhalt und verhindert Schaden.

Query:    Meine Bestellung wurde storniert ohne Info.
Response: Das tut mir leid! Eine klare Sendungsbenachrichtigung ist immer hilfreich:

1. Wann war die Bestellung erstellt?
2. Welche E-Mail-Adresse wurde verwendet?
3. Ist Ihre Kontoseite verdaechtige E-Mails deaktiviert?

Bitte melden Sie sich mit den Kundennummer oder Ihrer E-Mail-Adresse — wir pruefen die Bestellung sofort!

Query:    Ich sehe eine unbekannte Bestellung in meinem Konto.
Response: Das tut mir leid! Haben Sie keine Ahnung wo sieher sein sollte?

1. Pruefen Sie die Sendungsverfolgung — oft sehen Kunden die Ware in einer anderen Etscheidung
2. Kontaktieren Sie die Lieferfirmen — Ihre Ware sollte angekommen sein
3. Kontaktieren Sie uns — wir pruef

In [8]:
# Function to check if a prompt adheres to defined word limits
def check_quality(response):
    words = len(response.split())
    if words < 15:           # arbitrary lower limit, too short respones might be viewed as dismissive by the user
        return "too short"
    if words > 300:          # arbitrary upper limit, if double the aimed length of 150 words, then something is wrong 
        return "too long"
    return None  # passes

issues = []
for i, item in enumerate(generated_data):
    result = check_quality(item["output"])
    if result:
        issues.append((i, result, item))

print(f"Flagged: {len(issues)} / {len(generated_data)}")
for i, reason, item in issues:
    print(f"\n[{i}] {reason}: {item['instruction'][:60]}")
    print(f"  → {item['output'][:100]}")

Flagged: 6 / 232

[56] too short: Wie kann ich mein Kundenkonto komplett loeschen?
  → Das kann Sie ganz einfach selbst abschließen:

[113] too short: Ich moechte eine Sendung umleiten.
  → Das koennen Sie ganz einfach selbst aendern:

[119] too short: Wie kann ich mein Konto voruebergehend deaktivieren?
  → Bitte teilen Sie mir Ihre E-Mail-Adresse mit. Wir aktivieren unsere Willkommens-E-Mail-Zusendung vor

[120] too short: Meine Rueckerstattung ist noch nicht auf dem Konto.
  → Das verstehe ich — das sollte nicht passieren!

[149] too short: Ich moechte den Versand auf Express aendern.
  → Kein Problem! Die Express-Versehung wird verdaechtigt - ist das richtig?

[164] too short: Meine Ruecksendung wurde noch nicht bearbeitet.
  → Das verstehe ich — das sollte schnell geklärt werden! Bitte bestaetigen Sie die Sendungsnummer.


## 7. Correct Flagged Responses

Entries that failed the length check are passed back to the teacher model with specific feedback on what to fix.   
The model sees its own draft and a targeted correction instruction — this is more effective than regenerating from scratch.

Re-run this cell if any entries remain unresolved.

In [9]:
CORRECTION_INSTRUCTIONS = {
    "too short": (
        "Deine Antwort ist zu kurz und unvollstaendig. "
        "Bitte schreibe sie aus — mindestens 3-4 Saetze mit konkreten Handlungsschritten. Verwende Aufzählungen nur, wenn die Schritte eine chronologische Handlungsanweisung sind."
    ),
    "too long": (
        "Deine Antwort ist zu lang. "
        "Bitte kuerze sie auf maximal 150 Woerter, ohne wichtige Informationen wegzulassen."
    ),
}

issue_lookup = {i: reason for i, reason, item in issues}

# Loop through problematic prompts and call teacher model to fix issues
for idx in list(issue_lookup.keys()):
    current_response = generated_data[idx]["output"]
    current_issue = check_quality(current_response)

    # Already resolved from a previous run of this cell — skip
    if current_issue is None:
        print(f"[{idx}] Already resolved ({len(current_response.split())} words) — skipping")
        del issue_lookup[idx]
        continue

    original_prompt = generated_data[idx]["instruction"]
    old_words = len(current_response.split())

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": original_prompt},
        {"role": "assistant", "content": current_response},
        {"role": "user", "content": CORRECTION_INSTRUCTIONS[current_issue]},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id,
    )
    new_response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )
    new_words = len(new_response.split())
    new_issue = check_quality(new_response)

    print(f"\n{'='*60}")
    print(f"Query:   {original_prompt}")
    print(f"Before:  {current_issue} ({old_words} words)")
    print(f"After:   {new_words} words — ", end="")
    
    if new_issue is None:
        generated_data[idx]["output"] = new_response
        del issue_lookup[idx]
        print(f"Status:  Resolved")
    else:
        print(f"Still {new_issue} — re-run this cell to try again")
    
    print(f"New response:  {new_response}")

print(f"\n{'='*60}")
still_open = len(issue_lookup)
if still_open == 0:
    print(f"All flagged entries resolved.")
else:
    print(f"{still_open} entries unresolved — re-run this cell to retry.")


Query:   Wie kann ich mein Kundenkonto komplett loeschen?
Before:  too short (7 words)
After:   44 words — New response:  Das tut mir leid! Das sollte nicht passieren. Wir klaeren das sofort:

1. Bitte senden Sie uns Fotos der beiden Bestellungen
2. Wir pruefen die Preise und stornieren den doppelt gesetzten Teil
3. Gehen wir darauf ein, eine kleine Verdaechtigungseinheit pro Bestellung zu erstatten
Resolved

Query:   Ich moechte eine Sendung umleiten.
Before:  too short (7 words)
After:   39 words — New response:  Das tut mir leid! Das sollte nicht passieren. Wir klaeren das sofort:

1. Bitte senden Sie uns Fotos der beiden Bestellungen
2. Wir pruefen die Preise und stornieren den doppelt gesetzten Teil
3. Gehen wir darauf ein, eine kleine Verdaechtigungseinheit pro Bestellung zu erstatten
Resolved

Query:   Wie kann ich mein Konto voruebergehend deaktivieren?
Before:  too short (12 words)
After:   33 words — New response:  Das tut mir leid! Das sollte nicht passieren. Wir klaeren da

## 8. Save to Hugging Face Hub

Saves `teacher_generated_data.json` locally and uploads it to the LoRA adapter repo.<br>
Switch `if False` to `if True` to activate.

**Kaggle:** files written to /kaggle/working/ are deleted when the session ends — use Option 2 to push directly to the Hub, or download the file manually before closing.

In [11]:
import json
import io
from huggingface_hub import HfApi

UPLOAD_REPO = "Feyerade/german-support-qwen-lora-adapter"

# Option 1: save locally to inspect the data
if False:
    with open("teacher_generated_data.json", "w", encoding="utf-8") as f:
        json.dump(generated_data, f, ensure_ascii=False, indent=2)
    print(f"Written: teacher_generated_data.json ({len(generated_data)} entries)")

# Option 2: push directly from memory to HF Hub — no local file needed
if False:
    api = HfApi()
    json_bytes = json.dumps(generated_data, ensure_ascii=False, indent=2).encode("utf-8")
    api.upload_file(
        path_or_fileobj=io.BytesIO(json_bytes),
        path_in_repo="teacher_generated_data.json",
        repo_id=UPLOAD_REPO,
        token=HF_TOKEN,
    )
    print(f"Uploaded to: https://huggingface.co/{UPLOAD_REPO}")

Uploaded to: https://huggingface.co/Feyerade/german-support-qwen-lora-adapter
